In [1]:
!pip install nltk

  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached nltk-3.9.2-py3-none-any.whl (1.5 MB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)

   ------------- -------------------------- 1/3 [regex]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   ---------------------------------------- 3/3 [nltk]



In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\Users\brijr\anaconda3\envs\atlas\lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251
c:\Users\brijr\anaconda3\envs\atlas\lib\site-packages\pydantic\_internal\_config.py:383: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


In [2]:
df = pd.read_csv("IMDB.csv")
df = df.sample(500)
df.to_csv("data.csv", index= False)
df.head()

,review,sentiment
573,"This is superb - the acting wonderful, sets, c...",positive
992,Holy @#%& this movie was still warm and juicy ...,negative
886,I agree with most of the Columbo fans that thi...,negative
459,I believe an entire book can be written about ...,negative
806,"At the time, ""My Left Foot"" was the little mov...",positive


In [3]:
# data preprocessing

# define text preprocessing functions
def lemmatization(text):
    """Lemmatize the Text..."""
    lemmatizer = WordNetLemmatizer()
    text= text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove urls from the text."""
    url_pattern = re.compile(r'http?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)


def noramlize_text(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [4]:
df = noramlize_text(df)
df.head()

,review,sentiment
573,superb acting wonderful set clothes music stor...,positive
992,holy movie still warm juicy pile made with tri...,negative
886,agree columbo fan movie unnecessary change for...,negative
459,believe entire book written odyssey remake cla...,negative
806,time my left foot little movie could hugely po...,positive


In [5]:
df['sentiment'].value_counts()

sentiment
negative    267
positive    233
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive', 'negative'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].map({'positive' : 1, 'negative' : 0})
df.head()

,review,sentiment
573,superb acting wonderful set clothes music stor...,1
992,holy movie still warm juicy pile made with tri...,0
886,agree columbo fan movie unnecessary change for...,0
459,believe entire book written odyssey remake cla...,0
806,time my left foot little movie could hugely po...,1


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [17]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
import dagshub
mlflow.set_tracking_uri("")
dagshub.init(repo_owner='brij', repo_name='MLOPS-Capstone-proj', mlflow=True)

mlflow.set_experiment('Logistic regression Baseline')

2025-11-04 12:25:12,408 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/brij26/MLOPS-Capstone-proj "HTTP/1.1 200 OK"


Initialized MLflow to track repo "brij26/MLOPS-Capstone-proj"

2025-11-04 12:25:12,420 - INFO - Initialized MLflow to track repo "brij26/MLOPS-Capstone-proj"


Repository brij26/MLOPS-Capstone-proj initialized!

2025-11-04 12:25:12,422 - INFO - Repository brij26/MLOPS-Capstone-proj initialized!


<Experiment: artifact_location='mlflow-artifacts:/416b669f99374244bf3bd399b6c68b89', creation_time=1762237905709, experiment_id='0', last_update_time=1762237905709, lifecycle_stage='active', name='Logistic regression Baseline', tags={}>

In [20]:
import mlflow
import logging
import time
import os
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting mlflow run....")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters..")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("max_features", 100)
        mlflow.log_param("test_size" , 0.25)

        logging.info("Initializing LogisticRegression model....")
        model = LogisticRegression(max_iter=800) # Increase max_iter to prevent non convergence issues

        logging.info("fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete")

        logging.info("logging model parameters..")
        mlflow.log_param("model", "logistic regression")

        logging.info("Making predictions")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics.....")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model....")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"model training and logging completed in {end_time - start_time:.2f} seconds.")


        # print the results for verification 
        print(f"accuracy : {accuracy}")
        print(f"precision : {precision}")
        print(f"recall : {recall}")
        print(f"f1 score : {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2025-11-04 12:25:22,188 - INFO - Starting mlflow run....
2025-11-04 12:25:22,574 - INFO - Logging preprocessing parameters..
2025-11-04 12:25:24,088 - INFO - Initializing LogisticRegression model....
2025-11-04 12:25:24,088 - INFO - fitting the model...
2025-11-04 12:25:24,107 - INFO - Model training complete
2025-11-04 12:25:24,107 - INFO - logging model parameters..
2025-11-04 12:25:24,641 - INFO - Making predictions
2025-11-04 12:25:24,641 - INFO - Calculating evaluation metrics.....
2025-11-04 12:25:24,654 - INFO - Logging evaluation metrics...
2025-11-04 12:25:26,154 - INFO - Saving and logging the model....
c:\Users\brijr\anaconda3\envs\atlas\lib\site-packages\_distutils_hack\__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditiona

accuracy : 0.648
precision : 0.6545454545454545
recall : 0.5901639344262295
f1 score : 0.6206896551724138
